# CNN-LSTM EEG Classifier — Unassisted vs LLM-Assisted
**USC EEG Creativity Study**

Based on: Jiang et al. (2025) - *The cognitive impacts of large language model interactions on problem solving and decision making using EEG analysis*

Pipeline:
1. Load raw EEG sessions
2. Convert to spectrograms using STFT
3. Train CNN (spatial features) + LSTM (temporal features)
4. Classify: 0 = Unassisted, 1 = LLM-Assisted
5. Leave-One-Subject-Out cross-validation

## 0. Imports

In [1]:
import os
import glob
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import mne
mne.set_log_level('ERROR')

from scipy.signal import stft
from scipy import stats

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
print('All imports successful!')

PyTorch version: 2.12.0
CUDA available: False
Using device: cpu
All imports successful!


## 1. Configuration

In [2]:
BASE_PATH = "/Users/agastyabassi/Library/CloudStorage/OneDrive-SharedLibraries-UniversityofSouthernCalifornia/Athena Saghi - EEG Cleaning/clean"
OUTPUT_DIR = "./cnn_lstm_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NON_BRAIN    = ['A1', 'A2', 'REF', 'GND']
SEGMENT_SEC  = 4.0      # seconds per segment
OVERLAP      = 0.5      # 50% overlap between segments
N_FFT        = 256      # STFT window size
HOP_LENGTH   = 64       # STFT hop length
FMIN         = 1.0      # minimum frequency
FMAX         = 45.0     # maximum frequency

# Training
BATCH_SIZE   = 8
EPOCHS       = 50
LR           = 1e-3
DROPOUT      = 0.5

print('Configuration loaded!')
print(f'Device: {device}')

Configuration loaded!
Device: cpu


## 2. Load Files

In [3]:
all_files = glob.glob(os.path.join(BASE_PATH, '*.set'))
control_files   = sorted([f for f in all_files if '_control_'   in os.path.basename(f)])
treatment_files = sorted([f for f in all_files if '_treatment_' in os.path.basename(f)])

# Keep only participants with BOTH conditions
ctrl_subjects = set(os.path.basename(f).split('_')[0] for f in control_files)
trt_subjects  = set(os.path.basename(f).split('_')[0] for f in treatment_files)
common = ctrl_subjects & trt_subjects

control_files   = [f for f in control_files   if os.path.basename(f).split('_')[0] in common]
treatment_files = [f for f in treatment_files if os.path.basename(f).split('_')[0] in common]

print(f'Control files:   {len(control_files)}')
print(f'Treatment files: {len(treatment_files)}')
print(f'Participants:    {sorted(common)}')

Control files:   36
Treatment files: 38
Participants:    ['P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P20', 'P21', 'P22', 'P23', 'P4', 'P5', 'P6', 'P7', 'P9']


## 3. Feature Extraction — STFT Spectrograms

In [4]:
def extract_stft_segments(filepath, segment_sec=4.0, overlap=0.5,
                          n_fft=256, hop_length=64, fmin=1.0, fmax=45.0,
                          non_brain=NON_BRAIN):
    """
    Load EEG file, split into segments, compute STFT spectrograms.
    Returns: array of shape (n_segments, n_channels, n_freqs, n_times)
    """
    raw = mne.io.read_raw_eeglab(filepath, preload=True, verbose=False)
    
    # Remove non-brain channels
    drop = [ch for ch in non_brain if ch in raw.ch_names]
    if drop:
        raw.drop_channels(drop)
    
    sfreq    = raw.info['sfreq']
    data     = raw.get_data()  # (n_channels, n_times)
    n_ch     = data.shape[0]
    seg_len  = int(segment_sec * sfreq)
    step     = int(seg_len * (1 - overlap))
    
    # Normalize each channel to zero mean unit variance
    data = (data - data.mean(axis=1, keepdims=True)) / (data.std(axis=1, keepdims=True) + 1e-8)
    
    segments = []
    start = 0
    while start + seg_len <= data.shape[1]:
        seg = data[:, start:start+seg_len]  # (n_ch, seg_len)
        
        # Compute STFT per channel
        ch_specs = []
        for ch in range(n_ch):
            f, t, Zxx = stft(seg[ch], fs=sfreq, nperseg=n_fft, noverlap=n_fft-hop_length)
            # Keep only frequencies in range
            freq_mask = (f >= fmin) & (f <= fmax)
            spec = np.abs(Zxx[freq_mask])  # magnitude spectrogram
            # Log transform for better dynamic range
            spec = np.log1p(spec)
            ch_specs.append(spec)
        
        ch_specs = np.array(ch_specs)  # (n_ch, n_freqs, n_times)
        segments.append(ch_specs)
        start += step
    
    return np.array(segments), raw.ch_names  # (n_segments, n_ch, n_freqs, n_times)

print('STFT extraction function defined!')

# Test on one file
print('\nTesting on first control file...')
test_segs, test_chs = extract_stft_segments(control_files[0])
print(f'Segments shape: {test_segs.shape}')
print(f'Channels: {test_chs}')

STFT extraction function defined!

Testing on first control file...
Segments shape: (423, 19, 38, 20)
Channels: ['P3', 'C3', 'F3', 'Fz', 'F4', 'C4', 'P4', 'Cz', 'Fp1', 'Fp2', 'T7', 'P7', 'O1', 'O2', 'F7', 'F8', 'P8', 'T8', 'Pz']


## 4. Build Full Dataset

In [5]:
X_list      = []
y_list      = []
groups_list = []

print('Processing UNASSISTED files...')
for fpath in control_files:
    fname   = os.path.basename(fpath)
    subject = fname.split('_')[0]
    try:
        segs, _ = extract_stft_segments(fpath)
        for seg in segs:
            X_list.append(seg)
            y_list.append(0)
            groups_list.append(subject)
        print(f'  ✓ {fname} — {len(segs)} segments')
    except Exception as e:
        print(f'  ✗ {fname}: {e}')

print('\nProcessing LLM-ASSISTED files...')
for fpath in treatment_files:
    fname   = os.path.basename(fpath)
    subject = fname.split('_')[0]
    try:
        segs, _ = extract_stft_segments(fpath)
        for seg in segs:
            X_list.append(seg)
            y_list.append(1)
            groups_list.append(subject)
        print(f'  ✓ {fname} — {len(segs)} segments')
    except Exception as e:
        print(f'  ✗ {fname}: {e}')

# Find minimum shape and standardize
min_freqs = min(x.shape[1] for x in X_list)
min_times = min(x.shape[2] for x in X_list)
min_chs   = min(x.shape[0] for x in X_list)

X      = np.array([x[:min_chs, :min_freqs, :min_times] for x in X_list])
y      = np.array(y_list)
groups = np.array(groups_list)

print(f'\nFull dataset shape: {X.shape}')
print(f'Labels: {np.bincount(y)} (0=Unassisted, 1=LLM-Assisted)')
print(f'Total segments: {len(X)}')

Processing UNASSISTED files...
  ✓ P10_control_A3_postcleaning1.set — 423 segments
  ✓ P10_control_B2_postcleaning1.set — 778 segments
  ✓ P11_control_B1_postcleaning1.set — 461 segments
  ✓ P12_control_A3_postcleaning1.set — 591 segments
  ✓ P12_control_B1_postcleaning1.set — 872 segments
  ✓ P13_control_A2_postcleaning1.set — 441 segments
  ✓ P13_control_B3_postcleaning1.set — 545 segments
  ✓ P14_control_A4_postcleaning1.set — 372 segments
  ✓ P14_control_B4_postcleaning1.set — 384 segments
  ✓ P15_control_A2_postcleaning1.set — 655 segments
  ✓ P15_control_B1_postcleaning1.set — 420 segments
  ✓ P16_control_A2_postcleaning1.set — 205 segments
  ✓ P16_control_B3_postcleaning1.set — 413 segments
  ✓ P17_control_A2_postcleaning1.set — 534 segments
  ✓ P17_control_A3_postcleaning1.set — 390 segments
  ✓ P18_control_A3_postcleaning1.set — 309 segments
  ✓ P18_control_B2_postcleaning1.set — 745 segments
  ✓ P20_control_A2_postcleaning1.set — 223 segments
  ✓ P20_control_B1_postcleaning1.

## 5. CNN-LSTM Model Architecture

In [6]:
class EEGDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


class CNNLSTM(nn.Module):
    """
    CNN-LSTM for EEG spectrogram classification.
    Based on Jiang et al. (2025).
    
    Input: (batch, channels, freqs, times)
    """
    def __init__(self, n_channels, n_freqs, n_times, n_classes=2, dropout=0.5):
        super(CNNLSTM, self).__init__()
        
        # CNN for spatial-spectral feature extraction
        self.cnn = nn.Sequential(
            # Block 1
            nn.Conv2d(n_channels, 32, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),
            nn.Dropout2d(dropout * 0.5),
            
            # Block 2
            nn.Conv2d(32, 64, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(2, 2)),
            nn.Dropout2d(dropout * 0.5),
            
            # Block 3
            nn.Conv2d(64, 128, kernel_size=(3, 3), padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((4, n_times // 4)),
        )
        
        # Calculate CNN output size
        self.cnn_out_size = 128 * 4
        
        # LSTM for temporal feature extraction
        self.lstm = nn.LSTM(
            input_size=self.cnn_out_size,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            dropout=dropout,
            bidirectional=True
        )
        
        # Classifier
        self.classifier = nn.Sequential(
            nn.Linear(128 * 2, 64),  # *2 for bidirectional
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_classes)
        )
    
    def forward(self, x):
        # x: (batch, channels, freqs, times)
        batch_size = x.shape[0]
        n_times = x.shape[-1] // 4
        
        # CNN feature extraction
        cnn_out = self.cnn(x)  # (batch, 128, 4, n_times//4)
        
        # Reshape for LSTM: (batch, time_steps, features)
        cnn_out = cnn_out.permute(0, 3, 1, 2)  # (batch, time, filters, freqs)
        cnn_out = cnn_out.reshape(batch_size, n_times, -1)  # (batch, time, features)
        
        # LSTM temporal modeling
        lstm_out, (hn, _) = self.lstm(cnn_out)
        
        # Use last hidden state from both directions
        # hn shape: (num_layers*2, batch, hidden_size)
        hidden = torch.cat([hn[-2], hn[-1]], dim=1)  # (batch, hidden*2)
        
        # Classify
        out = self.classifier(hidden)
        return out


# Test model
n_ch, n_freqs, n_times = X.shape[1], X.shape[2], X.shape[3]
print(f'Input shape: channels={n_ch}, freqs={n_freqs}, times={n_times}')

test_model = CNNLSTM(n_ch, n_freqs, n_times, dropout=DROPOUT).to(device)
test_input = torch.FloatTensor(X[:2]).to(device)
test_out   = test_model(test_input)
print(f'Model output shape: {test_out.shape}')
print(f'Model parameters: {sum(p.numel() for p in test_model.parameters()):,}')

Input shape: channels=19, freqs=38, times=20
Model output shape: torch.Size([2, 2])
Model parameters: 1,167,554


## 6. Training Functions

In [7]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(y_batch).sum().item()
        total += y_batch.size(0)
    return total_loss / len(loader), correct / total


def eval_model(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_probs, all_preds, all_labels = [], [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            total_loss += loss.item()
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
            _, predicted = outputs.max(1)
            correct += predicted.eq(y_batch).sum().item()
            total += y_batch.size(0)
            all_probs.extend(probs)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    acc = correct / total
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except:
        auc = 0.5
    return total_loss / len(loader), acc, auc, all_preds, all_labels

print('Training functions defined!')

Training functions defined!


## 7. Leave-One-Subject-Out Cross-Validation

In [ ]:
logo = LeaveOneGroupOut()
unique_subjects = np.unique(groups)

fold_accs, fold_aucs = [], []
all_preds_loso, all_labels_loso = [], []

print('=== LEAVE-ONE-SUBJECT-OUT CROSS-VALIDATION ===')
print(f'Total segments: {len(X)}')
print(f'Subjects: {len(unique_subjects)}\n')

for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
    test_subj = groups[test_idx[0]]
    
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    train_dataset = EEGDataset(X_train, y_train)
    test_dataset  = EEGDataset(X_test,  y_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
    
    # Initialize model
    model     = CNNLSTM(n_ch, n_freqs, n_times, dropout=DROPOUT).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
    
    # Train
    best_val_acc = 0
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
        val_loss, val_acc, val_auc, _, _ = eval_model(model, test_loader, criterion)
        scheduler.step()
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= 15:  # early stopping
            break
    
    # Evaluate best model
    model.load_state_dict(best_model_state)
    _, test_acc, test_auc, preds, labels = eval_model(model, test_loader, criterion)
    
    fold_accs.append(test_acc)
    fold_aucs.append(test_auc)
    all_preds_loso.extend(preds)
    all_labels_loso.extend(labels)
    
    print(f'Fold {fold+1:2d} | Subject: {test_subj} | Acc: {test_acc:.3f} | AUC: {test_auc:.3f} | Epochs: {epoch+1}')

print(f'\n=== RESULTS ===')
print(f'Mean Accuracy: {np.mean(fold_accs):.3f} ± {np.std(fold_accs):.3f}')
print(f'Mean AUC:      {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}')
print(f'Chance level:  0.500')

=== LEAVE-ONE-SUBJECT-OUT CROSS-VALIDATION ===
Total segments: 33077
Subjects: 18



## 8. Results Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix
cm = confusion_matrix(all_labels_loso, all_preds_loso)
disp = ConfusionMatrixDisplay(cm, display_labels=['Unassisted', 'LLM-Assisted'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Confusion Matrix\n(Leave-One-Subject-Out)', fontweight='bold')

# Per-fold accuracy
axes[1].bar(range(len(fold_accs)), fold_accs, color='steelblue', alpha=0.8)
axes[1].axhline(0.5, color='red', linestyle='--', label='Chance (0.5)')
axes[1].axhline(np.mean(fold_accs), color='green', linestyle='--',
                label=f'Mean ({np.mean(fold_accs):.3f})')
axes[1].set_xlabel('Subject (Fold)', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Per-Subject Accuracy', fontweight='bold', fontsize=13)
axes[1].set_xticks(range(len(fold_accs)))
axes[1].set_xticklabels(unique_subjects, rotation=45, ha='right', fontsize=8)
axes[1].legend()
axes[1].set_ylim(0, 1.05)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'cnn_lstm_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: cnn_lstm_results.png')

print('\nClassification Report:')
print(classification_report(all_labels_loso, all_preds_loso,
                            target_names=['Unassisted', 'LLM-Assisted']))

## 9. Summary

In [ ]:
print('=' * 60)
print('CNN-LSTM CLASSIFIER SUMMARY')
print('=' * 60)
print(f'\nModel: CNN (spatial) + Bidirectional LSTM (temporal)')
print(f'Input: STFT spectrograms ({n_ch} channels x {n_freqs} freqs x {n_times} times)')
print(f'Validation: Leave-One-Subject-Out ({len(unique_subjects)} folds)')
print(f'Segments: {len(X)} total ({SEGMENT_SEC}s each, {int(OVERLAP*100)}% overlap)')
print(f'Chance level: 0.500')
print(f'\n--- Results ---')
print(f'Mean Accuracy: {np.mean(fold_accs):.3f} ± {np.std(fold_accs):.3f}')
print(f'Mean AUC:      {np.mean(fold_aucs):.3f} ± {np.std(fold_aucs):.3f}')
print(f'Best fold:     {max(fold_accs):.3f} (Subject {unique_subjects[np.argmax(fold_accs)]})')
print(f'Worst fold:    {min(fold_accs):.3f} (Subject {unique_subjects[np.argmin(fold_accs)]})')
print(f'\n--- Comparison ---')
print(f'Random Forest (band power): 69.4% accuracy, AUC=0.778')
print(f'CNN-LSTM (spectrograms):    {np.mean(fold_accs)*100:.1f}% accuracy, AUC={np.mean(fold_aucs):.3f}')
print(f'\n--- Output files ---')
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(f'  {f}')